# Part 1: Linear model

Pour la première deadline il faut rendre un fichier `y_pred.csv` qui sont les prédictions du modèle linéaire sur `X.csv` (dans *data/data_unlabeled/*)

1. Train/Validation Split
2. Baseline Model
3. Model Selection (tester plusieurs modèles)
4. Hyperparameter Tuning
5. Évaluation Finale sur Test Set
6. Prédictions sur Unlabeled Data

## Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, LassoLarsCV
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, GridSearchCV
from sklearn.metrics import mean_squared_error

import sys
sys.path.append('..')
from src.preprocessing import preprocess_pipeline

In [ ]:
X_train = pd.read_csv('../data/data_labeled/X_train.csv')
y_train = pd.read_csv('../data/data_labeled/y_train.csv')
X_test = pd.read_csv('../data/data_labeled/X_test.csv')
y_test = pd.read_csv('../data/data_labeled/y_test.csv')

# Vérification des dimensions avant prétraitement
print("Dimensions avant prétraitement:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

# Aligner les données 
X_train = X_train.loc[y_train.index]
X_test = X_test.loc[y_test.index]

# preprocessing
X_train_prep, scaler = preprocess_pipeline(X_train, fit_scaler=True)
X_test_prep, _ = preprocess_pipeline(X_test, scaler=scaler, fit_scaler=False)

# Vérification des dimensions après prétraitement
print("\nDimensions après prétraitement et alignement:")
print(f"X_train_prep: {X_train_prep.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test_prep: {X_test_prep.shape}")
print(f"y_test: {y_test.shape}")

## 1. Train - validation split
### Questions
- Quelle proportion train/validation ? (80/20 ou 90/10)
### Réponses
- Plus on a données de validation meilleur sera notre estimation mais moins de données pour s'entrainer. Vu qu'on a environ 1000 données il faut prendre 80-20

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
SHUFFLE = True

## 2. Baseline model

Il faut établir une référence, un modèle simple, basique par rapport auquel on va comparer les performances de notre modèle réel.
### Questions
- Lql choisir ?
- Doit-on faire déjà de la feature selection ?
- Sur quel data on évalue la baseline ?
- Quel metrics utiliser ? 

### Réponses
- Le plus basique : LinearRegression
- Pas encore, on utilise toutes les caractéristiques pour avoir un premier RMSE, avoir des données brut
- Entrainer sur X_train et évaluer sur X_val
- Les assistants ont dit qu'ils utiliseraient RMSE du coup la meme

In [ ]:
# Diviser les données en 80% de train - 20% de validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_prep, 
    y_train.values.ravel(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=SHUFFLE
)
# entrainement
baseline_model = LinearRegression()
baseline_model.fit(X_train_split, y_train_split)

# Prédictions sur l'ensemble de validation
y_val_pred = baseline_model.predict(X_val)
# Pour comparer, métrique sur l'ensemble d'entraînement
y_train_pred = baseline_model.predict(X_train_split)

val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
train_rmse = np.sqrt(mean_squared_error(y_train_split, y_train_pred))

print("Performance du modèle de base sur l'ensemble de validation:")
print(f"validation RMSE: {val_rmse:.4f}")
print("\nPerformance sur l'ensemble d'entraînement (pour détecter l'overfitting):")
print(f"train RMSE: {train_rmse:.4f}")

mse_scores = -cross_val_score(LinearRegression(),
                              X_train_prep,
                              y_train.values.ravel(),
                              cv=5,
                              scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(mse_scores)
print("CV RMSE per fold:", np.round(rmse_scores,4))
print("CV RMSE mean:", rmse_scores.mean(), "std:", rmse_scores.std())

train_sizes, train_scores, val_scores = learning_curve(LinearRegression(),
                                                       X_train_prep,
                                                       y_train.values.ravel(),
                                                       cv=5,
                                                       scoring='neg_mean_squared_error',
                                                       train_sizes=np.linspace(0.1,1.0,5))
train_rmse = np.sqrt(-train_scores)
val_rmse = np.sqrt(-val_scores)

plt.plot(train_sizes, train_rmse.mean(axis=1), label='train RMSE')
plt.plot(train_sizes, val_rmse.mean(axis=1), label='val RMSE')
plt.xlabel('Taille du train set')
plt.ylabel('RMSE')
plt.legend()
plt.show()

### Commentaires
Pas d'overfitting car le modèle ginéralise au moins auss bien que sur les données d'entrainement (différence de 0.0012).

Maintenant, tous nos modèles doivent faire bcp mieux que ca 

## 2. Model selection
**Objectif:** tester plusieurs modèles linéaires
### Questions
- Pq ne pas rester avec LinearRegression ?
- Comment choisir le param alpha ? 
- Pour chaque modèle, sur quoi évaluer
### Réponses 
- La régularisation des autres modèles permet de poser des coefficients sur les features qui nous interessent plus ou pas => meilleur généralisation
- Alpha faible -> peu de régularisation, alpha fort -> forte régularisation. On ne sait pas à l'avance lequel est le plus optimal. Tester plusieurs [0.01, 0.1, 1, 10, 100]
- Entrainer sur X_train et tester sur X_val, puis voir la RMSE de validation

In [ ]:
# modèles à tester
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=0.1)': Ridge(alpha=0.1),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Ridge (alpha=10.0)': Ridge(alpha=10.0),
    'Ridge (alpha=50.0)': Ridge(alpha=50.0),
    'Ridge (alpha=100.0)': Ridge(alpha=100.0),
    'Ridge (alpha=500.0)': Ridge(alpha=500.0),
    'Ridge (alpha=1000.0)': Ridge(alpha=1000.0),
    'Ridge (alpha=100000.0)': Ridge(alpha=100000.0),
    'Lasso (alpha=1.0)': Lasso(alpha=1.0),
    'Lasso (alpha=10.0)': Lasso(alpha=10.0),
    'Lasso (alpha=100.0)': Lasso(alpha=100.0),
    'LassoCV': LassoCV(random_state=RANDOM_STATE),
    'LassoLarsCV': LassoLarsCV()
}

# Dictionnaire pour stocker les résultats
results = {
    'train_rmse': [],
    'val_rmse': [],
    'model_names': []
}

# pour chaque modèle
for name, model in models.items():
    model.fit(X_train_split, y_train_split)
    y_train_pred = model.predict(X_train_split)
    y_val_pred = model.predict(X_val)
    
    # Calcul des métriques
    train_rmse = np.sqrt(mean_squared_error(y_train_split, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    
    # Stockage des résultats
    results['model_names'].append(name)
    results['train_rmse'].append(train_rmse)
    results['val_rmse'].append(val_rmse)
    
    print(f"\nRésultats pour {name}:")
    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Validation RMSE: {val_rmse:.4f}")

# Création d'un DataFrame pour une meilleure visualisation
results_df = pd.DataFrame({
    'Model': results['model_names'],
    'Train RMSE': results['train_rmse'],
    'Validation RMSE': results['val_rmse']
})

# Affichage des résultats triés par RMSE de validation
print("\nComparaison des modèles (triés par RMSE de validation):")
print(results_df.sort_values('Validation RMSE'))

In [ ]:
# Créer un meilleur graphique pour comparer les modèles
plt.figure(figsize=(15, 7))

# Trier les résultats par RMSE de validation
sorted_indices = np.argsort(results['val_rmse'])
model_names = np.array(results['model_names'])[sorted_indices]
train_rmse = np.array(results['train_rmse'])[sorted_indices]
val_rmse = np.array(results['val_rmse'])[sorted_indices]

# Créer les points et les lignes
plt.plot(model_names, train_rmse, 'bo-', label='Train RMSE', markersize=8)
plt.plot(model_names, val_rmse, 'ro-', label='Validation RMSE', markersize=8)

# Ajout des valeurs RMSE au-dessus des points
for i, (train, val) in enumerate(zip(train_rmse, val_rmse)):
    plt.text(i, train, f'{train:.4f}', ha='center', va='bottom')
    plt.text(i, val, f'{val:.4f}', ha='center', va='top')

# Personnalisation du graphique
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Modèles')
plt.ylabel('RMSE')
plt.title('Comparaison des performances des modèles (triés par RMSE de validation)')
plt.xticks(range(len(model_names)), model_names, rotation=45, ha='right')
plt.legend(loc='upper left')

# Ajuster les marges
plt.tight_layout()

# Ajouter une marge en haut pour les annotations
plt.margins(y=0.2)

plt.show()

# Afficher aussi un tableau avec les différences relatives par rapport au baseline
baseline_val_rmse = results_df[results_df['Model'] == 'Linear Regression']['Validation RMSE'].values[0]
results_df['Amélioration (%)'] = ((baseline_val_rmse - results_df['Validation RMSE']) / baseline_val_rmse * 100)
print("\nComparaison avec le baseline (Linear Regression):")
print(results_df.sort_values('Validation RMSE')[['Model', 'Validation RMSE', 'Amélioration (%)']])

### Commentaires
On voit que le Ridge apporte une amélioration sur le modele de base seulement à partir d'un Alpha supérieur à 1. le meilleur modèle se trouve entre le Ridge à alpha = 500 et alpha = 1000.

On voit que le RMSE passe de 0.0795 (baseline model) à 0.07895 (meilleur modele), c'est quand même tres faible
- Pq amélioration si faible ?
    - preprocessing déjà très bon / peu de feature redondantes / alpha testés pas dans la bonne range 

## 3. Hyperparamter tuning
On veut trouver le meilleur alpha, on a vu a la main ce que ca donnait maintenant on va passer à de l'automatisation via la cross-validation.

In [ ]:
param_grid = {
    'alpha': [1, 10, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
}

grid_search = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='neg_mean_squared_error',
    n_jobs=-1,  # Utiliser tous les CPU disponibles
    verbose=1
)

# Fit sur TOUT X_train_prep (pas juste le split)
grid_search.fit(X_train_prep, y_train.values.ravel())

# Récupérer les meilleurs résultats
best_alpha = grid_search.best_params_['alpha']
best_score_neg_mse = grid_search.best_score_
best_rmse_cv = np.sqrt(-best_score_neg_mse)

print("\n" + "=" * 60)
print("RÉSULTATS DE GRIDSEARCHCV")
print("=" * 60)
print(f"Meilleur alpha trouvé : {best_alpha}")
print(f"Meilleur RMSE (CV) : {best_rmse_cv:.6f}")
print()

# Afficher tous les résultats
results_cv = pd.DataFrame(grid_search.cv_results_)
results_cv['mean_rmse'] = np.sqrt(-results_cv['mean_test_score'])
results_cv['std_rmse'] = np.sqrt(results_cv['std_test_score'])

print("Détails de tous les alphas testés :")
print(results_cv[['param_alpha', 'mean_rmse', 'std_rmse']].sort_values('mean_rmse'))

# Visualiser la courbe alpha vs RMSE
plt.figure(figsize=(12, 6))
plt.plot(results_cv['param_alpha'], results_cv['mean_rmse'], 'b-o', linewidth=2, markersize=8)
plt.fill_between(
    results_cv['param_alpha'],
    results_cv['mean_rmse'] - results_cv['std_rmse'],
    results_cv['mean_rmse'] + results_cv['std_rmse'],
    alpha=0.3
)
plt.axvline(x=best_alpha, color='r', linestyle='--', label=f'Best alpha = {best_alpha}')
plt.xlabel('Alpha', fontsize=12)
plt.ylabel('RMSE (Cross-Validation)', fontsize=12)
plt.title('Impact de Alpha sur la Performance (5-Fold CV)', fontsize=14)
plt.xscale('log')  # Échelle log pour mieux voir
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Récupérer le meilleur modèle
best_model = grid_search.best_estimator_

print(f"\nMeilleur modèle sélectionné : Ridge(alpha={best_alpha})")

## 4. Feature importance
Analyse des coefficients

In [ ]:
# Récupérer les coefficients du meilleur modèle
coefficients = best_model.coef_
feature_names = X_train_prep.columns
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

# Trier par valeur absolue (les plus importantes en premier)
coef_df_sorted = coef_df.sort_values('Abs_Coefficient', ascending=False)

print("\nTop 15 features les plus importantes :")
print(coef_df_sorted.head(15).to_string(index=False))

# Visualisation : Top 10 features
plt.figure(figsize=(12, 8))
top_15 = coef_df_sorted.head(15)

colors = ['green' if x > 0 else 'red' for x in top_15['Coefficient']]
plt.barh(range(len(top_15)), top_15['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_15)), top_15['Feature'])
plt.xlabel('Coefficient', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Features les Plus Importantes (Ridge)', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)

# Légende
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Effet positif (↑ risque)'),
    Patch(facecolor='red', alpha=0.7, label='Effet négatif (↓ risque)')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

# Comparaison avec les découvertes de l'EDA
print("\n" + "=" * 60)
print("VALIDATION AVEC L'EDA")
print("=" * 60)
print("\nFeatures identifiées comme importantes dans l'EDA :")
print("- vitamin_D (corrélation négative)")
print("- cholesterol (corrélation positive)")
print("- blood_pressure (corrélation positive)")
print("\nCoefficients dans le modèle Ridge :")
for feat in ['vitamin D', 'cholesterol', 'blood pressure']:
    if feat in coef_df['Feature'].values:
        coef_val = coef_df[coef_df['Feature'] == feat]['Coefficient'].values[0]
        print(f"- {feat:20s} : {coef_val:+.6f}")

## 5. Evaluation finale sur Test set

In [ ]:
# Réentraîner le meilleur modèle sur TOUT le train set (train + validation réunis)
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train_prep, y_train.values.ravel())

print(f"\nModèle final réentraîné sur {len(X_train_prep)} samples")
print(f"Alpha utilisé : {best_alpha}")

# Prédire sur le test set
y_test_pred = final_model.predict(X_test_prep)

# Calculer RMSE sur test
test_rmse = np.sqrt(mean_squared_error(y_test.values.ravel(), y_test_pred))

# Calculer aussi RMSE sur train (pour détecter overfitting)
y_train_final_pred = final_model.predict(X_train_prep)
train_final_rmse = np.sqrt(mean_squared_error(y_train.values.ravel(), y_train_final_pred))

print(f"RMSE Train (final)      : {train_final_rmse:.6f}")
print(f"RMSE Validation (CV)    : {best_rmse_cv:.6f}")
print(f"RMSE Test               : {test_rmse:.6f}")
print()
print(f"Différence Val-Test     : {abs(test_rmse - best_rmse_cv):.6f}")
# Comparaison avec le baseline
baseline_test_rmse = np.sqrt(mean_squared_error(
    y_test.values.ravel(),
    LinearRegression().fit(X_train_prep, y_train.values.ravel()).predict(X_test_prep)
))

improvement = (baseline_test_rmse - test_rmse) / baseline_test_rmse * 100

print("\n" + "=" * 60)
print("COMPARAISON AVEC LE BASELINE")
print("=" * 60)
print(f"RMSE Baseline (LinearRegression) : {baseline_test_rmse:.6f}")
print(f"RMSE Final (Ridge)               : {test_rmse:.6f}")
print(f"Amélioration                     : {improvement:.2f}%")

# Visualisation : Prédictions vs Réel
plt.figure(figsize=(14, 6))

# Subplot 1 : Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_test.values.ravel(), y_test_pred, alpha=0.6, s=30)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Parfait')
plt.xlabel('Valeurs Réelles', fontsize=12)
plt.ylabel('Prédictions', fontsize=12)
plt.title('Prédictions vs Valeurs Réelles (Test Set)', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)

# Subplot 2 : Distribution des résidus
residuals = y_test.values.ravel() - y_test_pred
plt.subplot(1, 2, 2)
plt.hist(residuals, bins=30, alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Résidus (Réel - Prédit)', fontsize=12)
plt.ylabel('Fréquence', fontsize=12)
plt.title('Distribution des Résidus', fontsize=14)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Commentaire 
Différence Val-Test     : 0.001426. Excellent ! Le modèle généralise très bien.

COMPARAISON AVEC LE BASELINE
RMSE Baseline (LinearRegression) : 0.082254
RMSE Final (Ridge)               : 0.082356
Amélioration                     : -0.12%
Ca j'ai un peu du mal a y croire je comprends pas comment c'est possible.

## 6. Prédictions sur unlabaled

In [ ]:
# Charger les données unlabeled
X_unlabeled = pd.read_csv('../data/data_unlabeled/X.csv')
print(f"\nDonnées unlabeled chargées : {X_unlabeled.shape}")

# Preprocessing avec LE MÊME SCALER que train
print("\nApplication du preprocessing...")
X_unlabeled_prep, _ = preprocess_pipeline(
    X_unlabeled,
    scaler=scaler,  # IMPORTANT : Réutiliser le scaler fitté sur train !
    fit_scaler=False  # NE PAS refitter le scaler !
)

print(f"Après preprocessing : {X_unlabeled_prep.shape}")

# Vérifier que les colonnes correspondent
print("\nVérification de la cohérence des features...")
if list(X_unlabeled_prep.columns) == list(X_train_prep.columns):
    print("Les features correspondent parfaitement")
else:
    print("ATTENTION : Les features ne correspondent pas !")
    missing_in_unlabeled = set(X_train_prep.columns) - set(X_unlabeled_prep.columns)
    missing_in_train = set(X_unlabeled_prep.columns) - set(X_train_prep.columns)
    if missing_in_unlabeled:
        print(f"Manquant dans unlabeled : {missing_in_unlabeled}")
    if missing_in_train:
        print(f"Manquant dans train : {missing_in_train}")

# Faire les prédictions
print("\nGénération des prédictions...")
y_pred_unlabeled = final_model.predict(X_unlabeled_prep)

# Statistiques des prédictions
print(f"Nombre de prédictions : {len(y_pred_unlabeled)}")
print(f"Min    : {y_pred_unlabeled.min():.6f}")
print(f"Max    : {y_pred_unlabeled.max():.6f}")
print(f"Mean   : {y_pred_unlabeled.mean():.6f}")
print(f"Median : {np.median(y_pred_unlabeled):.6f}")
print(f"Std    : {y_pred_unlabeled.std():.6f}")

# Comparaison avec y_train (sanity check)
print("\n" + "=" * 60)
print("COMPARAISON AVEC Y_TRAIN (Sanity Check)")
print("=" * 60)
print(f"y_train - Mean   : {y_train.values.mean():.6f}")
print(f"y_pred  - Mean   : {y_pred_unlabeled.mean():.6f}")
print(f"y_train - Std    : {y_train.values.std():.6f}")
print(f"y_pred  - Std    : {y_pred_unlabeled.std():.6f}")

diff_mean = abs(y_train.values.mean() - y_pred_unlabeled.mean())
if diff_mean < 0.05:
    print("Distribution cohérente avec y_train")
else:
    print("Attention : Distribution très différente de y_train")

# Visualiser la distribution
plt.figure(figsize=(14, 6))

# Subplot 1 : Histogramme comparatif
plt.subplot(1, 2, 1)
plt.hist(y_train.values.ravel(), bins=30, alpha=0.6, label='y_train', color='blue', edgecolor='black')
plt.hist(y_pred_unlabeled, bins=30, alpha=0.6, label='y_pred (unlabeled)', color='orange', edgecolor='black')
plt.xlabel('Risque Cardiaque', fontsize=12)
plt.ylabel('Fréquence', fontsize=12)
plt.title('Distribution : y_train vs y_pred', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)

# Subplot 2 : Boxplot comparatif
plt.subplot(1, 2, 2)
data_to_plot = [y_train.values.ravel(), y_pred_unlabeled]
plt.boxplot(data_to_plot, labels=['y_train', 'y_pred (unlabeled)'])
plt.ylabel('Risque Cardiaque', fontsize=12)
plt.title('Boxplot Comparatif', fontsize=14)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Vérifier s'il y a des valeurs aberrantes
n_below_zero = (y_pred_unlabeled < 0).sum()
n_above_one = (y_pred_unlabeled > 1).sum()
print(f"Prédictions < 0   : {n_below_zero} ({n_below_zero/len(y_pred_unlabeled)*100:.2f}%)")
print(f"Prédictions > 1   : {n_above_one} ({n_above_one/len(y_pred_unlabeled)*100:.2f}%)")

## 7. Sauvegarde des données de prédictions

In [ ]:
# Chemin du fichier
output_path = '../results/predictions/y_pred.csv'
np.savetxt(output_path, y_pred_unlabeled, fmt='%.6f')
print(f"\nFichier sauvegardé : {output_path}")
with open(output_path, 'r') as f:
    lines = f.readlines()
    
    print(f"Nombre de lignes : {len(lines)}")
    print(f"Nombre attendu   : {len(X_unlabeled)}")
    
    if len(lines) == len(X_unlabeled):
        print("Nombre de lignes correct")
    else:
        print("ERREUR : Nombre de lignes incorrect !")
    
    print("\nAperçu du fichier :")
    print(f"Première ligne   : {lines[0].strip()}")
    print(f"Deuxième ligne   : {lines[1].strip()}")
    print(f"Avant-dernière   : {lines[-2].strip()}")
    print(f"Dernière ligne   : {lines[-1].strip()}")
    
    # Vérifier qu'il n'y a pas de header
    try:
        float(lines[0].strip())
        print("\nFormat correct : Pas de header détecté")
    except ValueError:
        print("\nERREUR : La première ligne n'est pas un nombre !")
    
    # Vérifier qu'il n'y a pas de guillemets
    if '"' in lines[0] or "'" in lines[0]:
        print("ERREUR : Des guillemets sont présents !")
    else:
        print("Format correct : Pas de guillemets")
    
    # Vérifier que tous les nombres sont valides
    try:
        all_values = [float(line.strip()) for line in lines]
        print("Toutes les lignes sont des nombres valides")
    except ValueError as e:
        print(f"ERREUR : Ligne invalide détectée - {e}")

print("\n" + "=" * 60)
print("RÉCAPITULATIF FINAL")
print("=" * 60)
print(f"Modèle final        : Ridge(alpha={best_alpha})")
print(f"RMSE Test           : {test_rmse:.6f}")
print(f"Prédictions         : {len(y_pred_unlabeled)} lignes")
print(f"Fichier sauvegardé  : {output_path}")